# MR2b — Type Count Store MapReduce
**Input**: `manhattan_liquor_stores_geocoded.csv`
**Output**: `type_count_store_mr.csv`

In [3]:
import h3
import pandas as pd
from collections import defaultdict

INPUT  = 'manhattan_liquor_stores_geocoded.csv'
OUTPUT = 'type_count_store_mr.csv'
H3_RES = 10

df = pd.read_csv(INPUT)
df['latitude']  = pd.to_numeric(df['latitude'],  errors='coerce')
df['longitude'] = pd.to_numeric(df['longitude'], errors='coerce')
df = df.dropna(subset=['latitude','longitude'])
df['method_of_operation'] = df['method_of_operation'].fillna('Unknown').str.strip().str.title()

print(f'Input rows: {len(df):,}')
print('Unique methods:', df['method_of_operation'].nunique())
df.head()

Input rows: 6,925
Unique methods: 1177


,serial_number,method_of_operation,license_type_code,county,premise_name,dba,premise_address,premise_city,certificate_number,license_expiration_date,latitude,longitude,status,error
0,1257248,Restaurant Serving Beer Cider & Wine Only,RW,NEW YORK,SOLO PIZZA INC,SOLO PIZZA,27 AVENUE B,NEW YORK,784148,2023-10-31T00:00:00.000,40.722365,-73.983177,success,NaN
1,1273203,Vessel Serving Liquor Beer Cider And Wine,VL,NEW YORK,HORNBLOWER CRUISES AND EVENTS LLC,MANHATTAN ELITE,PIER 62 CHELSEA PIERS,NEW YORK,803625,2023-10-31T00:00:00.000,40.748378,-74.008445,success,NaN
2,1320487,Unknown,AX,NEW YORK,581 R & V FOOD CORP,NaN,581 W 207TH ST,NEW YORK,935179,2023-10-31T00:00:00.000,40.866943,-73.920716,success,NaN
3,1337942,"Restaurant Serving Beer, Wine, Liquor And Cider",OP,NEW YORK,HSE ENTERPRISES LLC,THE PARLOUR ROOM,70 W 36TH ST,NEW YORK,890693,2023-10-31T00:00:00.000,40.750744,-73.986072,success,NaN
4,1330731,Importer,IL,NEW YORK,TAT AMERICA LLC,HOUSE OF BURGUNDY WINES AND SPIRITS,318 W 53RD ST,NEW YORK,935461,2023-12-31T00:00:00.000,40.764497,-73.985531,success,NaN


In [ ]:
# ── MAP ───────────────────────────────────────────────────────────────────────
def mapper(row):
    try:
        zone_id = h3.latlng_to_cell(float(row['latitude']), float(row['longitude']), H3_RES)
    except:
        return None, None
    key = (zone_id, str(row['method_of_operation']))
    return key, 1

mapped = [mapper(row) for _, row in df.iterrows()]
mapped = [(k,v) for k,v in mapped if k is not None]
print(f'Mapped pairs: {len(mapped):,}')

Mapped pairs: 6,925


In [ ]:
# ── REDUCE ────────────────────────────────────────────────────────────────────
acc = defaultdict(int)
for key, val in mapped:
    acc[key] += val

rows = [
    {'zone_id': zone_id, 'method_of_operation': method, 'count': cnt}
    for (zone_id, method), cnt in acc.items()
]

# ── OUTPUT ────────────────────────────────────────────────────────────────────
out = pd.DataFrame(rows).sort_values(['zone_id','count'], ascending=[True,False]).reset_index(drop=True)
out.to_csv(OUTPUT, index=False)

print(f'Output rows : {len(out):,}')
print(f'Unique zones: {out["zone_id"].nunique():,}')
print(f'Saved → {OUTPUT}')

# Sample: top methods per zone
print('\nSample zone breakdown:')
sample_zone = out['zone_id'].iloc[0]
print(out[out['zone_id']==sample_zone].to_string(index=False))
out.head(10)

Output rows : 6,576
Unique zones: 1,823
Saved → type_count_store_mr.csv

Sample zone breakdown:
        zone_id                            method_of_operation  count
8a2a1008804ffff      Tavern Serving Beer Cider Wine And Liquor      1
8a2a1008804ffff Restaurant Serving Beer, Wine, Liquor, & Cider      1
8a2a1008804ffff  Restaurant Serving Beer Cider Wine And Liquor      1


,zone_id,method_of_operation,count
0,8a2a1008804ffff,Tavern Serving Beer Cider Wine And Liquor,1
1,8a2a1008804ffff,"Restaurant Serving Beer, Wine, Liquor, & Cider",1
2,8a2a1008804ffff,Restaurant Serving Beer Cider Wine And Liquor,1
3,8a2a10088067fff,Grocery Store Selling Beer Cider And Wine Prod...,1
4,8a2a10088067fff,Restaurant Serving Beer Wine Cider & Liquor,1
5,8a2a10088067fff,Restaurant Serving Beer Cider And Wine,1
6,8a2a10088067fff,"Restaurant Serving Beer, Wine, & Cider",1
7,8a2a1008814ffff,"Restaurant Serving Beer, Cider And Wine",1
8,8a2a1008814ffff,"Restaurant Serving Liquor, Wine, Beer And Cider",1
9,8a2a1008814ffff,Restaurant Serving Beer & Cider,1
